# Random Grid Search

Baseline optimizer. See `docs/research.md` for methodology.

## Setup

In [4]:
import importlib
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

ensure_notebook_repo_context = importlib.import_module(
    "prediction_market_extensions.backtesting._notebook_support"
).ensure_notebook_repo_context
repo_root = ensure_notebook_repo_context()

## Configuration

In [5]:
import os
from decimal import Decimal

from dotenv import load_dotenv

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import BookReplay
from prediction_market_extensions.backtesting.data_sources import Book, Polymarket, Telonex
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


load_dotenv(repo_root / ".env")
telonex_api_key = os.environ.get("TELONEX_API_KEY", "")
if not telonex_api_key:
    raise RuntimeError("TELONEX_API_KEY is required in .env for Telonex API-first notebooks.")

name = "notebook_optimizer"

data = MarketDataConfig(
    platform=Polymarket,
    data_type=Book,
    vendor=Telonex,
    sources=(
        f"api:{telonex_api_key}",
        "local:/Volumes/LaCie/telonex_data",
    ),
)

base_replays = (
    BookReplay(market_slug="will-ludvig-aberg-win-the-2026-masters-tournament", token_index=0),
)

train_windows = (
    ParameterSearchWindow(
        name="train-2025-08-30-to-2026-04-07",
        start_time="2025-08-30T00:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
)

holdout_windows = (
    ParameterSearchWindow(
        name="holdout-2026-04-08",
        start_time="2026-04-08T00:00:00Z",
        end_time="2026-04-08T23:59:59Z",
    ),
)

strategy_spec = {
    "strategy_path": "strategies:BookDeepValueHoldStrategy",
    "config_path": "strategies:BookDeepValueHoldConfig",
    "config": {
        "trade_size": Decimal("5"),
        "entry_price_max": "__SEARCH__:entry_price_max",
        "single_entry": True,
    },
}

parameter_grid = {
    "entry_price_max": (0.70, 0.95),
}

execution = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

max_trials = 2
holdout_top_k = 1

parameter_search = ParameterSearchConfig(
    name=name,
    data=data,
    base_replays=base_replays,
    strategy_spec=strategy_spec,
    parameter_grid=parameter_grid,
    train_windows=train_windows,
    holdout_windows=holdout_windows,
    max_trials=max_trials,
    random_seed=7,
    holdout_top_k=holdout_top_k,
    initial_cash=100.0,
    probability_window=256,
    min_book_events=100,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=execution,
    sampler="random",
)

experiment = ParameterSearchExperiment(
    name=name,
    description="Deep-value random-grid search on Telonex L2 book data",
    parameter_search=parameter_search,
)

print(
    f"Configured random-grid L2 book search: {max_trials} trials over "
    f"{len(base_replays)} market(s), {len(train_windows)} train window(s), "
    f"{len(holdout_windows)} holdout window(s)."
)

Configured random-grid L2 book search: 2 trials over 1 market(s), 1 train window(s), 1 holdout window(s).


## Run optimizer

In [ ]:
import asyncio
from pprint import pprint

from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output
from prediction_market_extensions.backtesting.optimizers import run_parameter_search

with suppress_notebook_cell_output():
    summary = await asyncio.to_thread(
        run_parameter_search,
        parameter_search,
        evaluator=lambda backtest: backtest.run(),
    )

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

## Leaderboard

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {experiment.name} (random-grid)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

## Combined holdout chart

In [ ]:
import gc

from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting._prediction_market_backtest import MarketReportConfig
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)
from prediction_market_extensions.backtesting.prediction_market import finalize_market_results

selected_window = holdout_windows[0] if holdout_windows else train_windows[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

plot_panels = (
    "total_equity",
    "equity",
    "market_pnl",
    "periodic_pnl",
    "yes_price",
    "allocation",
    "total_drawdown",
    "drawdown",
    "total_rolling_sharpe",
    "rolling_sharpe",
    "total_cash_equity",
    "cash_equity",
    "monthly_returns",
    "total_brier_advantage",
    "brier_advantage",
)

report = MarketReportConfig(
    count_key="book_events",
    count_label="Book Events",
    pnl_label="PnL (USDC)",
    market_key="slug",
    summary_report=True,
    summary_report_path=f"output/{name}_holdout_joint_portfolio.html",
    summary_plot_panels=plot_panels,
)

backtest = build_parameter_search_window_backtest(
    config=parameter_search,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{name}:{selected_window.name}:selected-params",
    return_summary_series=True,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()
    finalize_market_results(
        name=backtest.name,
        results=results,
        report=report,
    )

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
num_markets = len(results)
print(f"Replayed {num_markets} joint-portfolio market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

del results, backtest, updated_html, html_snapshot
gc.collect()